In [0]:
import os
import time
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

import mlflow
import mlflow.spark
from mlflow.models.signature import infer_signature

CATALOG = "fahrlytics"
GOLD    = f"{CATALOG}.gold"
MODEL_NAME = f"{CATALOG}.models.fraud_detector_rf"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.models")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.models.mlflow_staging")

# Configure UC Volume staging path for MLflow SparkML models
UC_VOLUME_TMP = f"/Volumes/{CATALOG}/models/mlflow_staging"
os.environ["MLFLOW_DFS_TMP"] = UC_VOLUME_TMP

mlflow.set_experiment(f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/fahrlytics_fraud_detection")

print(f"Catalog            : {CATALOG}")
print(f"Source Feature Table: {GOLD}.payment_features")
print(f"MLflow Temp Volume : {UC_VOLUME_TMP}")
print(f"Target Model Name  : {MODEL_NAME}")


In [0]:
print("Loading Gold payment features...")

raw_df = spark.table(f"{GOLD}.payment_features")
df = raw_df.withColumn("label", F.col("is_fraud_label").cast("double"))

total_records = df.count()
fraud_count   = df.filter(F.col("label") == 1.0).count()
legit_count   = total_records - fraud_count
fraud_pct     = (fraud_count / total_records) * 100.0

print(f"Total Transactions : {total_records:,}")
print(f"Legitimate (0.0)   : {legit_count:,} ({100.0 - fraud_pct:.2f}%)")
print(f"Fraudulent (1.0)   : {fraud_count:,} ({fraud_pct:.2f}%)")

In [0]:
# Compute inverse class frequencies to penalize false negatives
# Weight for class 1 = legit_count / fraud_count
imbalance_ratio = float(legit_count) / float(fraud_count) if fraud_count > 0 else 1.0
print(f"Calculated Imbalance Ratio: 1 fraud case for every {imbalance_ratio:.2f} legit cases")

df_weighted = df.withColumn(
    "class_weight",
    F.when(F.col("label") == 1.0, F.lit(imbalance_ratio)).otherwise(F.lit(1.0))
)

In [0]:
# Use a stratified random split (80/20) with a fixed seed
train_df, test_df = df_weighted.randomSplit([0.8, 0.2], seed=42)

train_count = train_df.count()
test_count  = test_df.count()
print(f"Training set count : {train_count:,}")
print(f"Test set count     : {test_count:,}")

In [0]:
# Feature Engineering Pipeline Specification
categorical_cols = ["city", "payment_method", "device_type", "vehicle_type"]

numerical_cols = [
    "amount",
    "card_linked_accounts_count",
    "card_lifetime_tx_count",
    "seconds_since_last_payment",
    "trip_distance_km",
    "trip_duration_min",
    "surge_multiplier",
    "customer_rating",
    "account_age_days_at_tx",
    "amount_per_km",
    "city_avg_amount",
    "amount_to_city_avg_ratio",
    "tx_hour"
]

boolean_cols = [
    "is_late_night_tx",
    "is_round_amount",
    "is_credit_card",
    "is_paypal",
    "is_ghost_trip"
]

# Cast booleans to double for VectorAssembler
for b_col in boolean_cols:
    train_df = train_df.withColumn(b_col, F.col(b_col).cast("double"))
    test_df  = test_df.withColumn(b_col, F.col(b_col).cast("double"))

stages = []

# Index and One-Hot Encode Categorical Columns
for c in categorical_cols:
    indexer = StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    encoder = OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_vec"])
    stages += [indexer, encoder]

# Assemble and Scale
assembler_inputs = [f"{c}_vec" for c in categorical_cols] + numerical_cols + boolean_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="raw_features", handleInvalid="keep")
scaler    = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=False)

stages += [assembler, scaler]

# Random Forest Classifier with cost-sensitive weighting
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="class_weight",  # Imbalance compensation
    numTrees=60,
    maxDepth=8,
    seed=42
)
stages.append(rf)

pipeline = Pipeline(stages=stages)

In [0]:
evaluator_pr  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
evaluator_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
evaluator_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
evaluator_pre = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")

print("Training Imbalance-Weighted Random Forest Model...")

with mlflow.start_run(run_name="rf_fraud_imbalance_weighted") as run:
    run_id = run.info.run_id
    t0 = time.time()
    
    # Train the pipeline
    model = pipeline.fit(train_df)
    train_duration = time.time() - t0
    
    # Predict on test set
    predictions = model.transform(test_df)
    
    # Evaluate
    pr_auc  = evaluator_pr.evaluate(predictions)
    roc_auc = evaluator_roc.evaluate(predictions)
    f1      = evaluator_f1.evaluate(predictions)
    recall  = evaluator_rec.evaluate(predictions)
    prec    = evaluator_pre.evaluate(predictions)
    
    # Log parameters
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("num_trees", 60)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("imbalance_ratio_weight", round(imbalance_ratio, 2))
    mlflow.log_param("train_samples", train_count)
    mlflow.log_param("test_samples", test_count)
    
    # Log metrics
    mlflow.log_metric("test_area_under_pr", pr_auc)
    mlflow.log_metric("test_area_under_roc", roc_auc)
    mlflow.log_metric("test_f1_score", f1)
    mlflow.log_metric("test_weighted_recall", recall)
    mlflow.log_metric("test_weighted_precision", prec)
    mlflow.log_metric("training_duration_sec", train_duration)
    
    print("\n--- MODEL PERFORMANCE RESULTS (TEST SET) ---")
    print(f"  PR-AUC (Primary Metric) : {pr_auc:.4f}")
    print(f"  ROC-AUC                : {roc_auc:.4f}")
    print(f"  Weighted F1-Score      : {f1:.4f}")
    print(f"  Weighted Recall        : {recall:.4f}")
    print(f"  Weighted Precision     : {prec:.4f}")
    print(f"  Training Time          : {train_duration:.2f}s")
    
    # Log Model Pipeline with explicit dfs_tmpdir
    sample_input = train_df.select(categorical_cols + numerical_cols + boolean_cols).limit(5).toPandas()
    sample_pred  = predictions.select("prediction").limit(5).toPandas()
    signature    = infer_signature(sample_input, sample_pred)
    
    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="fraud_detector_pipeline",
        signature=signature,
        dfs_tmpdir=UC_VOLUME_TMP
    )
    print(f"\nModel successfully logged with Run ID: {run_id}")

In [0]:
print("\nConfusion Matrix on Test Set:")
predictions.groupBy("label", "prediction").count().show()

In [0]:
rf_stage = model.stages[-1]
importances = rf_stage.featureImportances.toArray()

print("Top 10 Fraud Predictors:")
feature_importance_tuples = sorted(zip(assembler_inputs, importances), key=lambda x: x[1], reverse=True)
for rank, (feat, score) in enumerate(feature_importance_tuples[:10], start=1):
    print(f"  {rank:>2}. {feat:<30} : {score * 100:.2f}%")

In [0]:
print(f"\nRegistering model to Unity Catalog: {MODEL_NAME}...")

model_uri = f"runs:/{run_id}/fraud_detector_pipeline"
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=MODEL_NAME
)

print(f"Model successfully registered as Version {registered_model.version} in Unity Catalog.")

In [0]:
print("Scoring recent transactions for operational review queue...")

scored_test = model.transform(test_df)

# Extract predicted fraud cases with probability vectors
flagged_cases = (
    scored_test.filter(F.col("prediction") == 1.0)
    .select(
        "payment_id",
        "city",
        "amount",
        "payment_method",
        "card_linked_accounts_count",
        "seconds_since_last_payment",
        "amount_per_km",
        "prediction"
    )
    .orderBy(F.col("amount").desc())
    .limit(10)
)

print("\nTop High-Risk Transactions Intercepted by Model:")
flagged_cases.show(truncate=False)